# plotter Module

This module produces 1-D slices for each kernel using both UCB and EI and displays suggested next points for each

## Imports

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from .base import AcquisitionStrategy

## Plotter

This class defines how the 1-D slices are created.

### plot_slice

For each axis, a grid is created with 300 points spread equally between the bounds and all other x values fixed at the suggested next point. The predict function is called to obtain the mean and standard deviation results for each set of x values, which are returned to their original scale. The grid is plotted with all raw data points, previously guessed points obtained from history.csv are highlighted, and the acquisition function is plotted.

### plot_model_comparison

This method orders the models by LML and prints the suggested next points for each and a combined plot showing the 1-D slices for each, along with the CV score.

In [ ]:
class Plotter:
    @staticmethod
    def plot_slice(
        optimizer,
        model, 
        result,
        title: str,
        acquisition: AcquisitionStrategy,
        ax_list = None,
        show: bool = True
    ):
        n_dims = optimizer.n_dims
        if ax_list is None:
            fig, ax_list = plt.subplots(1, n_dims, figsize=(max(4 * n_dims, 8), 4))
            ax_list = np.atleast_1d(ax_list)
            is_standalone = True
        else:
            is_standalone = False
        
        y_best_orig = optimizer.raw_y.max()

        # History Data
        hist_X = None
        if optimizer.history is not None and not optimizer.history.empty:
            data_vals = optimizer.history.values
            hist_X = data_vals[:, :n_dims]
            hist_y = data_vals[:, -1]

        for d, ax in enumerate(ax_list):
            grid = np.linspace(optimizer.bounds[0], optimizer.bounds[1], 300)
            X_grid = np.tile(result.next_coords, (grid.size, 1))
            X_grid[:, d] = grid

            # Predictions (uses Prediction dataclass)
            mu_z, std_z = optimizer.predict(model, X_grid, return_std=True, return_scaled=True)
            
            # Unscale for plotting
            mu_model = mu_z * optimizer.scaler.scale_[0] + optimizer.scaler.mean_[0]
            std_model = std_z * optimizer.scaler.scale_[0]
            
            upper_model = mu_model + 1.96 * std_model
            lower_model = mu_model - 1.96 * std_model

            if optimizer.log_transform_y:
                mu_plot = np.exp(mu_model)
                upper_plot = np.exp(upper_model)
                lower_plot = np.exp(lower_model)
            else:
                mu_plot = mu_model
                upper_plot = upper_model
                lower_plot = lower_model

            # Plot Model
            ax.fill_between(grid, lower_plot, upper_plot, color='tab:blue', alpha=0.15, zorder=0)
            ax.plot(grid, mu_plot, lw=2, color='tab:blue', zorder=1, label="Mean Prediction")
            ax.scatter(optimizer.X[:, d], optimizer.raw_y, s=25, color='k', zorder=2, label="Current Data")
            
            if hist_X is not None:
                ax.scatter(hist_X[:, d], hist_y, s=25, color='purple', zorder=3, label="History")
            
            ax.axhline(y_best_orig, color="k", ls=":", lw=1.5, alpha=0.5)

            # Plot Acquisition
            ax2 = ax.twinx()
            acq_z = optimizer._calc_score(X_grid, model, acquisition)
            
            if acq_z.max() > 0:
                acq_norm = acq_z / acq_z.max()
                plot_range = mu_plot.max() - mu_plot.min()
                if plot_range == 0: plot_range = 1.0
                acq_vis = acq_norm * plot_range * 0.4 + mu_plot.min()
            else:
                acq_vis = acq_z

            ax2.plot(grid, acq_vis, color="tab:red", lw=2, linestyle='--', zorder=4, label=acquisition.name)
            ax2.axvline(result.next_coords[d], color='tab:red', ls='-', lw=1, alpha=0.8, zorder=5)

            if d == 0:
                ax.set_ylabel("Target Output", color='tab:blue')
                h1, l1 = ax.get_legend_handles_labels()
                h2, l2 = ax2.get_legend_handles_labels()
                by_label = dict(zip(l1+l2, h1+h2))
                ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize='x-small', framealpha=0.9)
            
            ax.set_title(f"Dim {d+1}", fontsize=10)
            ax2.set_yticks([]) 

        if is_standalone and show:
            plt.suptitle(title, fontsize=14)
            plt.tight_layout()
            plt.show()

    @staticmethod
    def plot_model_comparison(optimizer, models_dict, acquisition, optimizer_strategy, df_ranks=None):
        if df_ranks is None:
            actual_model = models_dict
            model_name = "NN Ensemble" if hasattr(optimizer, 'build_ensemble') else "Single Model"
            models_dict = {model_name: actual_model}
            df_ranks = pd.DataFrame([{'name': model_name, 'mean_score': 0.0}])

        n_models = len(models_dict)
        n_dims = optimizer.n_dims
        
        fig, axes = plt.subplots(n_models, n_dims, figsize=(max(4 * n_dims, 10), 3.5 * n_models), constrained_layout=False)
        if n_models == 1: axes = axes[np.newaxis, :]
        if n_dims == 1: axes = axes[:, np.newaxis]

        print(f"\n[Plotter] Model Analysis & Suggestions ({acquisition.name})")
        print(f"{'Model Name':<25} | {'CV Score':<8} | {'Suggested Point'}")
        print("-" * 90)

        for i, row in df_ranks.iterrows():
            model_name = row['name']
            model = models_dict[model_name]
            score_display = f"{row['mean_score']:.3f}" if row['mean_score'] != 0.0 else "N/A"
            
            res = optimizer.suggest(model, acquisition, optimizer_strategy)
            
            coords_str = "-".join([f"{x:.6f}" for x in res.next_coords])
            print(f"{model_name:<25} | {score_display:<8} | {coords_str}")
            
            row_text = f"{model_name}"
            if score_display != "N/A": row_text += f"\nCV: {score_display}"
            
            axes[i, 0].text(-0.30, 0.5, row_text, transform=axes[i, 0].transAxes, 
                            va='center', ha='right', fontsize=10, fontweight='bold')

            Plotter.plot_slice(
                optimizer, model, res, "", acquisition, 
                ax_list=axes[i], show=False
            )

        plt.suptitle(f"Optimization Landscape ({acquisition.name})", fontsize=16)
        plt.subplots_adjust(wspace=0.4, hspace=0.4, top=0.9)
        plt.show()